In [1]:
import pandas as pd
from Bio import Entrez
Entrez.email = 'dmalzl@cemm.oeaw.ac.at'
Entrez.api_key = '746c80043a6e2ceeb01ae587f8ab1a8ace09'


srauid_map = pd.read_csv(
    'archs4_accession_to_uid.tsv',
    sep = '\t'
)

# may contain duplicates
srauid_map.drop_duplicates(inplace = True)
srauid_map

,accession,uid,database
0,SRX185895,243394,sra
1,SRX185896,243395,sra
2,SRX185897,243396,sra
3,SRX185898,243397,sra
4,SRX185899,243398,sra
...,...,...,...
722660,SRX185248,242745,sra
722661,SRX185249,242746,sra
722662,SRX185250,242747,sra
722663,SRX185251,242748,sra


In [ ]:
from metadatamapping import dbutils

uid_batches = it.batched(
    srauid_map.uid,
    n = 50000
)
accession_frames = []
n = 0
for uid_list in uid_batches:
    print(f'retrieving uids {n} to {n + len(uid_list)}')
    n += len(uid_list)
    web_env_info = create_webenv(uid_list, 'sra')
    print(web_env_info)
    
    accessions = dbutils.data_from_esummary(
        web_env_info,
        'sra',
        dbutils.parse_esummary_response,
        len(uid_list),
        accession_matchers
    )
    accession_frames.append(accessions)

accessions = pd.concat(accession_frames)

In [30]:
accessions

,experiment,sample,study,biosample,bioproject,geo
0,SRX185895,SRS362050,SRP007525,SAMN01163911,PRJNA138597,GSM1000981
1,SRX185896,SRS362051,SRP007525,SAMN01163912,PRJNA138597,GSM1000982
2,SRX185897,SRS362052,SRP007525,SAMN01163913,PRJNA138597,GSM1000983
3,SRX185898,SRS362053,SRP007525,SAMN01163914,PRJNA138597,GSM1000984
4,SRX185899,SRS362054,SRP007525,SAMN01163915,PRJNA138597,GSM1000985
...,...,...,...,...,...,...
22420,SRX185248,SRS361742,SRP015668,SAMN01163627,PRJNA174763,GSM999587
22421,SRX185249,SRS361743,SRP015668,SAMN01163628,PRJNA174763,GSM999588
22422,SRX185250,SRS361744,SRP015668,SAMN01163629,PRJNA174763,GSM999589
22423,SRX185251,SRS361745,SRP015668,SAMN01163630,PRJNA174763,GSM999590


In [22]:
def determine_accession_type(accession):
    types = {
        'experiment': 'SRX',
        'biosample': 'SAMN',
        'geo': 'GSM'
    }
    
    for accession_type, accession_prefix in types.items():
        if accession.startswith(accession_prefix):
            return accession_type
    

srauid_map['accession_type'] = srauid_map.accession.apply(
    lambda x: determine_accession_type(x)
)
srauid_map

,accession,uid,database,accession_type
0,SRX185895,243394,sra,experiment
1,SRX185896,243395,sra,experiment
2,SRX185897,243396,sra,experiment
3,SRX185898,243397,sra,experiment
4,SRX185899,243398,sra,experiment
...,...,...,...,...
722660,SRX185248,242745,sra,experiment
722661,SRX185249,242746,sra,experiment
722662,SRX185250,242747,sra,experiment
722663,SRX185251,242748,sra,experiment


In [23]:
merged_groups = []
for accession_type, group in srauid_map.groupby('accession_type'):
    group = group.rename(
        columns = {'accession': accession_type}
    )
    
    merged_group = group.merge(
        accessions,
        on = accession_type,
        how = 'inner'
    )
    merged_groups.append(merged_group)
    
merged = pd.concat(merged_groups)
merged

,biosample,uid,database,accession_type,experiment,sample,study,bioproject,geo
0,SAMN02383463,2398516,sra,biosample,SRX1671895,SRS1369054,SRP031885,PRJNA224140,GSM1249874
1,SAMN02383461,2398517,sra,biosample,SRX1671896,SRS1369053,SRP031885,PRJNA224140,GSM1249875
2,SAMN02383456,2398518,sra,biosample,SRX1671897,SRS1369052,SRP031885,PRJNA224140,GSM1249876
3,SAMN02383457,2398519,sra,biosample,SRX1671898,SRS1369051,SRP031885,PRJNA224140,GSM1249877
4,SAMN02383460,2398520,sra,biosample,SRX1671899,SRS1369050,SRP031885,PRJNA224140,GSM1249878
...,...,...,...,...,...,...,...,...,...
27,SAMN29786852,23023289,sra,geo,SRX16311437,SRS13950350,SRP386774,PRJNA859437,GSM6341224
28,SAMN29786851,23023290,sra,geo,SRX16311438,SRS13950351,SRP386774,PRJNA859437,GSM6341225
29,SAMN29786850,23023291,sra,geo,SRX16311439,SRS13950352,SRP386774,PRJNA859437,GSM6341226
30,SAMN29786849,23023292,sra,geo,SRX16311440,SRS13950353,SRP386774,PRJNA859437,GSM6341227


In [24]:
merged.to_csv(
    'sample_accessions.tsv.gz',
    sep = '\t',
    index = False,
    compression = 'gzip'
)

In [20]:
merged = pd.read_csv(
    'sample_accessions.tsv.gz',
    sep = '\t',
    compression = 'gzip'    
)

In [3]:
# eLink does not work for id lists larger than 1000
# see this issue https://github.com/biopython/biopython/issues/1559
# need to batch them locally
import itertools as it


def any_links(link_item):
    return True if link_item['LinkSetDb'] else False


def get_links(link_item):
    # some accessions not be released so we can't link them
    if any_links(link_item):
        dbto_ids = link_item['LinkSetDb'][0]['Link']
        
    else:
        dbto_ids = [{'Id': None}]
        
    dbfrom_ids = link_item['IdList']
    
    links = [
        [dbfrom_id, dbto_id['Id']] for dbfrom_id, dbto_id in it.product(dbfrom_ids, dbto_ids)
    ]
    
    return links
        

def parse_elink_response(elink_response_handle):
    elink_response = Entrez.read(elink_response_handle)
    linked_ids = []
    for link_item in elink_response:
        links = get_links(
            link_item
        )
        linked_ids.extend(links)
    
    return linked_ids
    

def link_ids(uid_list, dbfrom, dbto):
    response_handle = Entrez.elink(
        dbfrom = dbfrom,
        db = dbto,
        id = uid_list
    )

    links = parse_elink_response(
        response_handle
    )
    
    return links

    
def link_sra_to_biosample(sra_uids):
    linked_ids = []
    n = 0
    dbfrom = 'sra'
    dbto = 'biosample'
    for id_chunk in it.batched(sra_uids, n = 1000):
        print(f'linking uids {n} to {n + len(id_chunk)}')
        n += len(id_chunk)
        
        links = retry(
            link_ids,
            id_chunk,
            dbfrom = dbfrom,
            dbto = dbto
        )
    
        linked_ids.extend(
            links
        )
    
    linked_ids = pd.DataFrame(
        linked_ids,
        columns = [dbfrom, dbto]
    )
    return linked_ids

In [ ]:
linked_uids = link_sra_to_biosample(srauid_map.uid)
linked_uids

In [36]:
linked_uids.to_csv(
    'sra_to_biosample.csv',
    index = False
)

In [14]:
import numpy as np


linked_uids = pd.read_csv(
    'sra_to_biosample.csv.gz',
    compression = 'gzip'
)
linked_uids = linked_uids.dropna()
linked_uids['biosample'] = linked_uids.biosample.astype(int)
linked_uids

,sra,biosample
0,243394,1163911
1,243395,1163912
2,243396,1163913
3,243397,1163914
4,243398,1163915
...,...,...
722428,242745,1163627
722429,242746,1163628
722430,242747,1163629
722431,242748,1163630


In [17]:
from xml.etree import ElementTree
from collections import defaultdict


def node_parser(node, parse_func, match_string):
    match = False
    if not node.tag == match_string:
        return match, dict()
    
    match = True
    return match, parse_func(node)


def parse_title(node):
    return {'title': node.text}
        
    
def parse_attribute(node):
    harmonized_name = node.attrib.get('harmonized_name')
    attribute_name = node.attrib.get('attribute_name')
    name = harmonized_name if harmonized_name else attribute_name
    return {f'attribute|{name}': node.text}


def parse_organism(node):
    return {'organism': ';'.join(node.attrib[k] for k in ['taxonomy_name', 'taxonomy_id'])}


def parse_biosample_uid(node):
    return {'biosample': node.attrib['id']}


def combine_attribute_keys(metadata):
    attribute_kv_list = []
    revised_metadata = dict()
    for key, value in metadata.items():
        if not key.startswith('attribute'):
            revised_metadata[key] = value
            continue
            
        name = key.split('|')[-1]
        attribute_kv_list.append(f'{name}: {value}')
    
    revised_metadata['attribute'] = '; '.join(attribute_kv_list)
    return revised_metadata


def extract_metadata(xmltree, metadata_parsers):
    metadata = dict()
    for node in xmltree.iter():
        for match_string, parse_function in metadata_parsers.items():
            match, parse_result = node_parser(
                node, 
                parse_function, 
                match_string
            )
            
            if match:
                metadata.update(parse_result)
                break
                
    return combine_attribute_keys(metadata)


def add_parsing_results_to_metadata(parsed_metadata, metadata):
    for key in metadata.keys():
        metadata[key].append(
            parsed_metadata.get(key)
        )
        

def parse_biosample_esummary_response(response_handle, metadata, metadata_parsers):
    for summary in Entrez.read(response_handle)['DocumentSummarySet']['DocumentSummary']:
        xmltree = ElementTree.fromstring(summary['SampleData'])
        parsed_metadata = extract_metadata(
            xmltree,
            metadata_parsers
        )
        add_parsing_results_to_metadata(
            parsed_metadata,
            metadata
        )


metadata_parsers = {
    'Title': parse_title,
    'Attribute': parse_attribute,
    'Organism': parse_organism,
    'BioSample': parse_biosample_uid
}


biosample_batches = it.batched(
    linked_uids.biosample,
    n = 50000
)
metadata_frames = []
n = 0
for uid_list in biosample_batches:
    print(f'retrieving uids {n} to {n + len(uid_list)}')
    n += len(uid_list)
    web_env_info = create_webenv(uid_list, 'biosample')
    print(web_env_info)
    
    metadata = get_data_from_esummary(
        web_env_info, 
        'biosample', 
        parse_biosample_esummary_response, 
        len(uid_list),
        metadata_parsers
    )
    metadata_frames.append(metadata)

metadata = pd.concat(metadata_frames)

retrieving uids 0 to 50000
{'QueryKey': '1', 'WebEnv': 'MCID_655714994ac1f824cd3d17b4'}
retrieving uids 100000 to 150000
{'QueryKey': '1', 'WebEnv': 'MCID_655715d022cf3030690be185'}
retrieving uids 150000 to 200000
{'QueryKey': '1', 'WebEnv': 'MCID_6557166cb4cc234ff22a2b75'}
retrieving uids 200000 to 250000
{'QueryKey': '1', 'WebEnv': 'MCID_655716ddb4cc234ff22a2ce6'}
retrieving uids 250000 to 300000
{'QueryKey': '1', 'WebEnv': 'MCID_655717525e9c76586032b298'}
retrieving uids 300000 to 350000
{'QueryKey': '1', 'WebEnv': 'MCID_655717b0b4cc234ff22a2f4c'}
retrieving uids 350000 to 400000
{'QueryKey': '1', 'WebEnv': 'MCID_65571813c93597127f343df2'}
retrieving uids 400000 to 450000
{'QueryKey': '1', 'WebEnv': 'MCID_65571874b4cc234ff22a31b0'}
retrieving uids 450000 to 500000
{'QueryKey': '1', 'WebEnv': 'MCID_655718dab4cc234ff22a3314'}
retrieving uids 500000 to 550000
{'QueryKey': '1', 'WebEnv': 'MCID_6557193ca21ef36d661657f2'}
retrieving uids 550000 to 600000
{'QueryKey': '1', 'WebEnv': 'MCID

In [18]:
metadata.to_csv('biosample_metadata.csv', index = False)

In [19]:
metadata

,title,attribute,organism,biosample
0,OCI-LY1_48hrs_mRNAseq_3x_siNT_R1,source_name: Human DLBCL cel line; treatment: ...,Homo sapiens;9606,1163911
1,OCI-LY1_48hrs_mRNAseq_3x_siNT_R2,source_name: Human DLBCL cel line; treatment: ...,Homo sapiens;9606,1163912
2,OCI-LY1_48hrs_mRNAseq_3x_siNT_R3,source_name: Human DLBCL cel line; treatment: ...,Homo sapiens;9606,1163913
3,OCI-LY1_48hrs_mRNAseq_3x_siBCL6_R1,source_name: Human DLBCL cel line; treatment: ...,Homo sapiens;9606,1163914
4,OCI-LY1_48hrs_mRNAseq_3x_siBCL6_R2,source_name: Human DLBCL cel line; treatment: ...,Homo sapiens;9606,1163915
...,...,...,...,...
22217,Parkinson's Disease PD 13,source_name: Brain cortex; tissue: Human brain...,Homo sapiens;9606,1163627
22218,Parkinson's Disease PD 14,source_name: Brain cortex; tissue: Human brain...,Homo sapiens;9606,1163628
22219,Parkinson's Disease PD 15,source_name: Brain cortex; tissue: Human brain...,Homo sapiens;9606,1163629
22220,Parkinson's Disease PD 16,source_name: Brain cortex; tissue: Human brain...,Homo sapiens;9606,1163630


In [21]:
import certifi
import urllib3
from io import BytesIO


study_ids = merged.study.unique()
metasra_data = []
urllib3.disable_warnings()
print(f'retrieving {len(study_ids)} studies')
for i, study_id in enumerate(study_ids):
    # cert_reqs = 'CERT_NONE' not recommended but needed due to unresolvable SSLError
    http = urllib3.PoolManager(
        ca_certs = certifi.where(),
        cert_reqs = 'CERT_NONE'                      
    )

    # retrieving number of studies that match search criteria
    r = http.request(
        'GET', 
        'http://metasra.biostat.wisc.edu/api/v01/samples.csv',
        fields = {
            'study': study_id,
            'species': 'human', 
            'assay': 'RNA-seq', 
            'limit': 10000,
            'skip': 0
        }
    )
    if not i + 1 % 1000:
        print(i)

    study_data = pd.read_csv(BytesIO(r.data))
    metasra_data.append(study_data)

metasra = pd.concat(metasra_data)

retrieving 20217 studies
1
1001
2001
3001
4001
5001
6001
7001
8001
9001
10001
11001
12001
13001
14001
15001
16001
17001
18001
19001
20001


/tmp/ipykernel_57092/74238611.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metasra = pd.concat(metasra_data)


In [22]:
metasra = pd.concat(metasra_data)
metasra

/tmp/ipykernel_57092/1472980570.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metasra = pd.concat(metasra_data)


,study_id,study_title,sample_id,sample_name,sample_type,sample_type_confidence,mapped_ontology_ids,mapped_ontology_terms,raw_SRA_metadata,sample_species,assay
0,SRP031885,Brd4 and JMJD6-associated Anti-pause Enhancers...,SRS1369056,HEK293T cell,cell line,1.000000,"CL:0000010, CVCL:0045, EFO:0001182, EFO:000072...","cultured cell, HEK293, transfection, kidney",cell line: Human Embryonic Kidney 293 cells; s...,human,RNA-seq
1,SRP031885,Brd4 and JMJD6-associated Anti-pause Enhancers...,SRS1369051,HEK293T cell,cell line,1.000000,"CL:0000010, CVCL:0045, EFO:0001182, EFO:000072...","cultured cell, HEK293, transfection, kidney",cell line: Human Embryonic Kidney 293 cells; s...,human,RNA-seq
2,SRP031885,Brd4 and JMJD6-associated Anti-pause Enhancers...,SRS1369055,HEK293T cell,cell line,1.000000,"CL:0000010, CVCL:0045, EFO:0001182, EFO:000072...","cultured cell, HEK293, transfection, kidney",cell line: Human Embryonic Kidney 293 cells; s...,human,RNA-seq
3,SRP031885,Brd4 and JMJD6-associated Anti-pause Enhancers...,SRS1369050,HEK293T cell,cell line,1.000000,"CL:0000010, CVCL:0045, EFO:0001182, EFO:000072...","cultured cell, HEK293, transfection, kidney",cell line: Human Embryonic Kidney 293 cells; s...,human,RNA-seq
4,SRP031885,Brd4 and JMJD6-associated Anti-pause Enhancers...,SRS1369059,HEK293T cell,cell line,1.000000,"CL:0000010, CVCL:0045, EFO:0001182, EFO:000072...","cultured cell, HEK293, transfection, kidney",cell line: Human Embryonic Kidney 293 cells; s...,human,RNA-seq
...,...,...,...,...,...,...,...,...,...,...,...
28,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361730,Brain cortex,tissue,0.945379,"DOID:14330, EFO:0002508, EFO:0003534, UBERON:0...","Parkinson's disease, dorsal telencephalon, mal...",disease state: Parkinson's Disease PD; gender:...,human,RNA-seq
29,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361740,Brain cortex,tissue,0.945379,"DOID:14330, EFO:0002508, EFO:0003534, UBERON:0...","Parkinson's disease, dorsal telencephalon, mal...",disease state: Parkinson's Disease PD; gender:...,human,RNA-seq
30,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361739,Brain cortex,tissue,0.945379,"DOID:14330, EFO:0002508, EFO:0003534, UBERON:0...","Parkinson's disease, dorsal telencephalon, mal...",disease state: Parkinson's Disease PD; gender:...,human,RNA-seq
31,SRP015668,aSyn polyA-RNAseq in PD and unaffected cortica...,SRS361732,Brain cortex,tissue,0.945379,"DOID:14330, EFO:0002508, EFO:0003534, UBERON:0...","Parkinson's disease, dorsal telencephalon, mal...",disease state: Parkinson's Disease PD; gender:...,human,RNA-seq


In [23]:
metasra.to_csv(
    'metasra_metadata.csv',
    index = False
)